# 🎙️ AI Voice Studio: Expressive Multilingual Voice Cloning
**Powered by Resemble AI Chatterbox & Gradio**

This notebook sets up a complete, production-quality AI voice studio directly in Google Colab.
It features zero-shot voice cloning, expressive speech (emotions, paralinguistics), and multi-language support.

### Features
* **Multilingual Cloning**: Clone voices in 20+ languages.
* **Expressive Delivery**: Use paralinguistic tags like `[laugh]` or `[sigh]`.
* **Smart Chunking**: Process long texts seamlessly.
* **Modern UI**: Full-featured Gradio web interface.

**Instructions:**
1. Go to **Runtime -> Change runtime type** and ensure **Hardware accelerator** is set to **T4 GPU** (or better).
2. Run all cells sequentially.
3. Click the public Gradio link generated at the bottom to access your Voice Studio.

### Expected GPU/VRAM Requirements
* **GPU**: NVIDIA T4 (or higher).
* **VRAM**: ~15GB available. The combined loaded models (Chatterbox Multilingual V3 & Turbo) will fit comfortably within Colab's standard T4 environment. Memory cleanup functions are provided in the UI for long sessions.

### Known Limitations
* The `[laugh]`, `[sigh]`, and other paralinguistic tags are natively supported only by the Turbo engine (English). For other languages, the system uses parameter adjustments (CFG/Exaggeration) to mimic expression.
* Uploaded reference clips should ideally be 5–15 seconds of clean, noise-free speech.

### Troubleshooting
* **CUDA Out of Memory:** Click the `🧹 Clear GPU Memory` button in the UI, or restart the runtime (`Runtime > Restart runtime`).
* **Generation takes too long:** Ensure you are connected to a GPU. If the top-right corner says "RAM" instead of "T4", change your runtime type.
* **Gradio link doesn't appear:** Sometimes the Gradio share server times out on the free tier. Try running the very last cell again.


In [ ]:
# CELL 2: Environment & GPU detection
import torch
import sys

print("=== System Information ===")
print(f"Python Version: {sys.version.split(' ')[0]}")
print(f"PyTorch Version: {torch.__version__}")

if torch.cuda.is_available():
    print(f"CUDA: Available")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ WARNING: GPU not detected. Generating audio on CPU will be extremely slow.")
    print("Please go to Runtime > Change runtime type and select a GPU (e.g., T4).")


In [ ]:
# CELL 3: Install dependencies
# 1. Uninstall pre-installed hf-gradio to prevent conflicts with newer Gradio versions
!pip uninstall -y hf-gradio

# 2. Install base PyTorch audio dependencies
!pip install --upgrade torch torchaudio --index-url https://download.pytorch.org/whl/cu121

# 3. Install the Voice Engine (Chatterbox), Gradio, and Audio utils
# We require gradio>=5.0.0 per project specifications
!pip install chatterbox-tts "gradio>=5.0.0" librosa soundfile scipy numpy


In [ ]:
# CELL 4: Verify package versions
import gradio as gr
import torchaudio
import librosa

print(f"Gradio version: {gr.__version__}")
print(f"Torchaudio version: {torchaudio.__version__}")
print(f"Librosa version: {librosa.__version__}")


In [ ]:
# CELL 5: Import libraries
import os
import gc
import re
import time
import datetime
import traceback
import tempfile
import uuid

import numpy as np
import torch
import torchaudio
import soundfile as sf
import librosa
import gradio as gr

# Import Chatterbox models
from chatterbox.tts import ChatterboxTTS
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
from chatterbox.tts_turbo import ChatterboxTurboTTS


In [ ]:
# CELL 6: Configuration
OUTPUT_DIR = "/content/voice_studio_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Global device set to: {DEVICE}")

# Environment variables
os.environ["GRADIO_SHARE"] = "1"


In [ ]:
# CELL 7: Language map
# Chatterbox Multilingual V3 supported languages
LANGUAGE_MAP = {
    "English 🇺🇸": "en",
    "Hindi 🇮🇳": "hi",
    "Spanish 🇪🇸": "es",
    "French 🇫🇷": "fr",
    "German 🇩🇪": "de",
    "Japanese 🇯🇵": "ja",
    "Korean 🇰🇷": "ko",
    "Chinese 🇨🇳": "zh",
    "Arabic 🇸🇦": "ar",
    "Italian 🇮🇹": "it",
    "Portuguese 🇵🇹": "pt",
    "Russian 🇷🇺": "ru",
    "Dutch 🇳🇱": "nl",
    "Polish 🇵🇱": "pl",
    "Swedish 🇸🇪": "sv",
    "Turkish 🇹🇷": "tr",
    "Finnish 🇫🇮": "fi",
    "Greek 🇬🇷": "el",
    "Danish 🇩🇰": "da",
    "Norwegian 🇳🇴": "no",
    "Swahili 🇰🇪": "sw",
    "Hebrew 🇮🇱": "he",
    "Malay 🇲🇾": "ms"
}


In [ ]:
# CELL 8: Expression presets
# The expression system uses Chatterbox Turbo's paralinguistic tags for English 
# and CFG/Exaggeration tuning for the Multilingual model.

EXPRESSION_PRESETS = {
    "Neutral": {"tag": "", "exaggeration": 0.5, "cfg_weight": 0.5},
    "Calm": {"tag": "[sigh]", "exaggeration": 0.4, "cfg_weight": 0.4},
    "Conversational": {"tag": "[chuckle]", "exaggeration": 0.6, "cfg_weight": 0.5},
    "Happy": {"tag": "[laugh]", "exaggeration": 0.7, "cfg_weight": 0.4},
    "Serious": {"tag": "[clear throat]", "exaggeration": 0.4, "cfg_weight": 0.6},
    "Dramatic": {"tag": "[gasp]", "exaggeration": 0.8, "cfg_weight": 0.3},
    "Energetic": {"tag": "", "exaggeration": 0.9, "cfg_weight": 0.4},
}

def apply_expression_preset(text, preset_name, intensity, is_turbo=False):
    """
    Modifies text (for Turbo) or parameters based on expression preset.
    """
    preset = EXPRESSION_PRESETS.get(preset_name, EXPRESSION_PRESETS["Neutral"])
    
    # Scale exaggeration based on intensity (0-100)
    # Default intensity 50 leaves it at preset default.
    intensity_scale = intensity / 50.0 
    
    final_exag = min(1.0, preset["exaggeration"] * intensity_scale)
    final_cfg = preset["cfg_weight"]
    
    if is_turbo and preset["tag"]:
        # Prepend tag for Turbo
        text = f"{preset['tag']} {text}"
        
    return text, final_exag, final_cfg


In [ ]:
# CELL 9: Audio preprocessing utilities
def preprocess_reference_audio(audio_path, target_sr=24000):
    """
    Prepares reference audio for the model.
    Converts to mono, resamples, normalizes, and trims silence.
    """
    if not audio_path or not os.path.exists(audio_path):
        return None
        
    try:
        # Load audio (automatically resamples if sr parameter is provided)
        y, sr = librosa.load(audio_path, sr=target_sr, mono=True)
        
        # Trim leading and trailing silence
        y_trimmed, index = librosa.effects.trim(y, top_db=30)
        
        # Normalize audio peak to -1 dB to prevent clipping
        peak = np.abs(y_trimmed).max()
        if peak > 0:
            y_normalized = y_trimmed * (10 ** (-1 / 20)) / peak
        else:
            y_normalized = y_trimmed
            
        # Save to a temporary file
        temp_dir = tempfile.gettempdir()
        out_path = os.path.join(temp_dir, f"ref_{uuid.uuid4().hex[:8]}.wav")
        sf.write(out_path, y_normalized, target_sr)
        
        return out_path
    except Exception as e:
        print(f"Error preprocessing audio: {e}")
        traceback.print_exc()
        return audio_path # Fallback to original


In [ ]:
# CELL 10: Model loading
# Global cache for models to avoid reloading
MODELS = {
    "multilingual": None,
    "turbo": None
}

def load_models():
    """
    Loads models into VRAM. Run once at startup.
    """
    global MODELS
    print("Loading models... This may take a minute.")
    
    try:
        # Load Multilingual V3
        if MODELS["multilingual"] is None:
            print("Loading Chatterbox Multilingual V3...")
            MODELS["multilingual"] = ChatterboxMultilingualTTS.from_pretrained(
                device=DEVICE, 
                t3_model="v3"
            )
            
        # Load Turbo (for English with Paralinguistics)
        if MODELS["turbo"] is None:
            print("Loading Chatterbox Turbo...")
            MODELS["turbo"] = ChatterboxTurboTTS.from_pretrained(
                device=DEVICE
            )
            
        print("✅ Models loaded successfully.")
    except Exception as e:
        print(f"❌ Error loading models: {e}")
        traceback.print_exc()

def clear_gpu_memory():
    """
    Clears PyTorch CUDA cache.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        return "🧹 GPU memory cleared."
    return "No GPU to clear."

# Load models immediately when cell runs
load_models()


In [ ]:
# CELL 11: Voice generation engine
def generate_voice_engine(text, ref_audio, language_id, preset, intensity, temperature, top_p, seed):
    """
    Core generation function wrapping the Chatterbox model.
    """
    if not text.strip():
        raise gr.Error("Please enter some text.")
    if not ref_audio:
        raise gr.Error("Please upload or record a reference voice.")
    if language_id not in LANGUAGE_MAP.values():
        raise gr.Error("Selected language is not supported.")
        
    # Preprocess audio (trim, normalize)
    processed_ref = preprocess_reference_audio(ref_audio)
    if not processed_ref:
        raise gr.Error("Failed to process reference audio.")

    # Determine which model to use
    is_turbo = (language_id == "en")
    model = MODELS["turbo"] if is_turbo else MODELS["multilingual"]
    
    if model is None:
        raise gr.Error("Model is not loaded. Please re-run the Model Loading cell.")

    # Apply expression preset
    proc_text, exag, cfg = apply_expression_preset(text, preset, intensity, is_turbo)
    
    # Set seed if provided
    if seed != -1:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    start_time = time.time()
    
    try:
        with torch.inference_mode():
            if is_turbo:
                # signature: (self, text, repetition_penalty=1.2, min_p=0.0, top_p=0.95, audio_prompt_path=None, exaggeration=0.0, cfg_weight=0.0, temperature=0.8, top_k=1000, norm_loudness=True)
                wav = model.generate(
                    text=proc_text,
                    audio_prompt_path=processed_ref,
                    exaggeration=exag,
                    cfg_weight=cfg,
                    temperature=temperature,
                    top_p=top_p
                )
            else:
                # signature: (self, text, language_id, audio_prompt_path=None, exaggeration=0.5, cfg_weight=0.5, temperature=0.8, repetition_penalty=1.2, min_p=0.05, top_p=1.0)
                wav = model.generate(
                    text=proc_text,
                    language_id=language_id,
                    audio_prompt_path=processed_ref,
                    exaggeration=exag,
                    cfg_weight=cfg,
                    temperature=temperature,
                    top_p=top_p
                )
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"Generation failed: {str(e)}")
        
    gen_time = time.time() - start_time
    
    # Save output
    timestamp = datetime.datetime.now().strftime("%Y_%m_%d_%H%M%S")
    out_path = os.path.join(OUTPUT_DIR, f"voice_{timestamp}.wav")
    
    # Wav is returned as a torch tensor, save using torchaudio
    # Ensure wav is a 2D tensor [channels, frames] as expected by torchaudio
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    torchaudio.save(out_path, wav, model.sr)
    
    return out_path, f"{gen_time:.2f}s", "Turbo (English)" if is_turbo else f"Multilingual ({language_id})"


In [ ]:
# CELL 12: Long-text chunking
def split_text(text, max_chars=200):
    """
    Splits long text into sentences/chunks to avoid memory issues and timeouts.
    """
    # Simple regex to split by sentence boundaries
    sentences = re.split(r'(?<=[.!?]) +', text.strip())
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chars:
            current_chunk += (sentence + " ")
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            # If a single sentence is longer than max_chars, we just have to append it
            current_chunk = sentence + " "
            
    if current_chunk:
        chunks.append(current_chunk.strip())
        
    return [c for c in chunks if c]

def generate_voice_chunked(text, ref_audio, language_id, preset, intensity, temperature, top_p, seed, progress=gr.Progress()):
    """
    Handles long text by splitting it, generating audio for each chunk, and concatenating.
    """
    if not text.strip():
        raise gr.Error("Please enter some text.")
        
    chunks = split_text(text)
    
    if len(chunks) == 1:
        # If it's short, just use the normal generation
        progress(0, desc="Generating...")
        return generate_voice_engine(chunks[0], ref_audio, language_id, preset, intensity, temperature, top_p, seed)
        
    all_wavs = []
    total_time = 0
    model_used = ""
    
    for i, chunk in enumerate(chunks):
        progress((i) / len(chunks), desc=f"Generating chunk {i+1} / {len(chunks)}...")
        
        # Advance seed for variation across chunks if a specific seed was given
        current_seed = seed + i if seed != -1 else -1
        
        out_path, time_str, m_used = generate_voice_engine(
            chunk, ref_audio, language_id, preset, intensity, temperature, top_p, current_seed
        )
        
        total_time += float(time_str.replace("s", ""))
        model_used = m_used
        
        # Load the generated chunk
        y, sr = librosa.load(out_path, sr=None)
        all_wavs.append(y)
        
        # Add a tiny bit of silence between chunks (e.g., 200ms)
        silence = np.zeros(int(sr * 0.2))
        all_wavs.append(silence)
        
    progress(1.0, desc="Concatenating chunks...")
    
    # Concatenate all audio arrays
    final_audio = np.concatenate(all_wavs)
    
    # Save the final concatenated audio
    timestamp = datetime.datetime.now().strftime("%Y_%m_%d_%H%M%S")
    final_out_path = os.path.join(OUTPUT_DIR, f"voice_{timestamp}.wav")
    
    # Determine correct SR based on model used
    final_sr = MODELS["turbo"].sr if "Turbo" in model_used else MODELS["multilingual"].sr
    sf.write(final_out_path, final_audio, final_sr)
    
    return final_out_path, f"{total_time:.2f}s", f"{model_used} (Chunked)"


In [ ]:
# CELL 13: Gradio UI & Launch
custom_css = """
.gradio-container { font-family: 'Inter', sans-serif; }
.header-text { text-align: center; margin-bottom: 20px; }
.header-text h1 { color: #2d3748; font-weight: 800; font-size: 2.5em; margin-bottom: 0.2em; }
.header-text p { color: #718096; font-size: 1.1em; }
.box-section { border: 1px solid #e2e8f0; border-radius: 12px; padding: 20px; margin-bottom: 20px; background: #f8fafc; }
.gen-button { font-size: 1.2em; padding: 15px; font-weight: bold; background: linear-gradient(90deg, #4F46E5 0%, #7C3AED 100%); color: white !important; border: none !important; }
.gen-button:hover { background: linear-gradient(90deg, #4338CA 0%, #6D28D9 100%); }
"""

with gr.Blocks(title="AI Voice Studio", css=custom_css, theme=gr.themes.Soft()) as demo:
    gr.HTML("""
    <div class="header-text">
        <h1>🎙️ VOICE STUDIO</h1>
        <p>Multilingual AI Voice Cloning & Expressive Speech</p>
    </div>
    """)
    
    with gr.Row():
        # LEFT COLUMN - INPUTS
        with gr.Column(scale=1):
            with gr.Group(elem_classes="box-section"):
                gr.Markdown("### 👤 Reference Voice")
                ref_audio = gr.Audio(
                    label="Upload or Record (WAV/MP3/M4A)", 
                    type="filepath", 
                    sources=["upload", "microphone"]
                )
                gr.Markdown("*Note: Use only voice recordings that you own or have permission to clone.*", elem_classes="text-xs text-gray-500")

            with gr.Group(elem_classes="box-section"):
                gr.Markdown("### 🌐 Language")
                language = gr.Dropdown(
                    choices=list(LANGUAGE_MAP.keys()),
                    value="English 🇺🇸",
                    label="Language Selection"
                )
                gr.Markdown("*Turbo model with paralinguistic tags is automatically used for English.*")

            with gr.Group(elem_classes="box-section"):
                gr.Markdown("### 📝 Script")
                text_input = gr.Textbox(
                    label="Text to Synthesize",
                    placeholder="Enter the text here. Long texts will be automatically chunked...\nTry typing: \"[laugh] That's so funny!\"",
                    lines=5,
                    max_lines=15
                )
                
            with gr.Group(elem_classes="box-section"):
                gr.Markdown("### 🎭 Expression System")
                with gr.Row():
                    preset = gr.Dropdown(
                        choices=list(EXPRESSION_PRESETS.keys()),
                        value="Neutral",
                        label="Delivery Style / Preset"
                    )
                    intensity = gr.Slider(
                        minimum=0, maximum=100, value=50, step=1,
                        label="Expression Intensity (50 = natural)"
                    )

            with gr.Accordion("⚙️ Advanced Settings", open=False):
                temperature = gr.Slider(minimum=0.1, maximum=2.0, value=0.8, step=0.1, label="Temperature (Randomness)")
                top_p = gr.Slider(minimum=0.1, maximum=1.0, value=0.95, step=0.05, label="Top P")
                seed = gr.Number(value=-1, label="Seed (-1 for random)", precision=0)
                
                with gr.Row():
                    random_seed_btn = gr.Button("🎲 Random Seed")
                    clear_mem_btn = gr.Button("🧹 Clear GPU Memory")

        # RIGHT COLUMN - OUTPUT
        with gr.Column(scale=1):
            with gr.Group(elem_classes="box-section"):
                gr.Markdown("### 🔊 Generation")
                gen_btn = gr.Button("GENERATE VOICE", variant="primary", elem_classes="gen-button")
                
                gr.Markdown("### Output")
                output_audio = gr.Audio(label="Audio Output", interactive=False, type="filepath")
                
                with gr.Row():
                    gen_time = gr.Textbox(label="Generation Time", interactive=False)
                    model_used = gr.Textbox(label="Model Used", interactive=False)

    # Event Handlers
    random_seed_btn.click(
        fn=lambda: -1,
        outputs=seed
    )
    
    clear_mem_btn.click(
        fn=clear_gpu_memory,
        outputs=gr.Textbox(label="Memory Status", visible=True)
    )

    # We need a small proxy function to resolve the language key to value because Gradio passing dict is tricky
    def proxy_generate(t, r, lang_map, lang_key, p, i, temp, tp, s):
        lang_id = lang_map.get(lang_key, "en")
        return generate_voice_chunked(t, r, lang_id, p, i, temp, tp, s)
        
    # Wire up the generate button
    gen_btn.click(
        fn=proxy_generate,
        inputs=[
            text_input,
            ref_audio,
            gr.State(LANGUAGE_MAP),
            language,
            preset,
            intensity,
            temperature,
            top_p,
            seed
        ],
        outputs=[output_audio, gen_time, model_used]
    )

print("Starting Gradio...")
share = os.environ.get("GRADIO_SHARE", "1") == "1"
demo.launch(share=share, inline=True)
